In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv("/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/interim/clean.csv", parse_dates=["date"])
EXPECTED_COLUMNS = ["date", "base", "quote", "rate"]

EXPECTED_DTYPES = {
    "date": "datetime64[ns]",
    "base": "object",   # or "category"
    "quote": "object",  # or "category"
    "rate": "float64",
}

ALLOWED_BASE = {"SAR"}
ALLOWED_QUOTE = {"AED", "CNY", "EUR", "GBP", "USD"}

def validate_fx_schema(df: pd.DataFrame) -> list[str]:
    errors = []

    # 1. Columns
    missing = set(EXPECTED_COLUMNS) - set(df.columns)
    extra = set(df.columns) - set(EXPECTED_COLUMNS)
    if missing:
        errors.append(f"Missing columns: {sorted(missing)}")
    if extra:
        errors.append(f"Unexpected columns: {sorted(extra)}")

    # 2. Dtypes
    for col, expected in EXPECTED_DTYPES.items():
        if col not in df.columns:
            continue
        actual = str(df[col].dtype)
        # allow category for base/quote
        if expected == "object" and actual == "category":
            continue
        if actual != expected:
            errors.append(f"{col}: expected {expected}, got {actual}")

    # 3. Nulls
    if df[EXPECTED_COLUMNS].isna().any().any():
        null_cols = df[EXPECTED_COLUMNS].isna().sum()
        null_cols = null_cols[null_cols > 0].to_dict()
        errors.append(f"Null values found: {null_cols}")

    # 4. Base / quote values
    if not df["base"].isin(ALLOWED_BASE).all():
        bad = df.loc[~df["base"].isin(ALLOWED_BASE), "base"].unique()
        errors.append(f"Invalid base values: {bad}")

    if not df["quote"].isin(ALLOWED_QUOTE).all():
        bad = df.loc[~df["quote"].isin(ALLOWED_QUOTE), "quote"].unique()
        errors.append(f"Invalid quote values: {bad}")

    # 5. Rate
    if not (df["rate"] > 0).all():
        bad = df.loc[df["rate"] <= 0, ["date", "base", "quote", "rate"]]
        errors.append(f"Non-positive rates: {len(bad)} rows")

    # 6. Uniqueness
    dup_mask = df.duplicated(subset=["date", "base", "quote"], keep=False)
    if dup_mask.any():
        errors.append(f"Duplicate (date, base, quote) keys: {dup_mask.sum()} rows")

    # 7. Date range
    if df["date"].isna().any():
        errors.append("Invalid/unparseable dates found")

    return errors


# After cleaning:
errors = validate_fx_schema(df)

if errors:
    print("Schema validation failed:")
    for e in errors:
        print("-", e)
        df.to_csv("/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/interim/clean_invalid.csv", index=False)
else:
    print("Schema validation passed.")
    df.to_csv("/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/interim/clean_validated.csv", index=False)

Schema validation passed.
